# FIRELINE — Kabupaten/Kota Drill-Down

Working notebook for Phase 1 onward. This notebook is the single working surface for validating the new administrative boundaries, normalizing population context, assigning grid-days, producing kabupaten summaries, and preparing dashboard outputs.

The existing grid-day priority score is kept unchanged.

## Working rules

- Primary boundaries come from `data/additions/*.geojson`.
- `kode_kk` is the candidate canonical kabupaten ID.
- Grid-day assignment uses the representative observed point.
- Population is contextual and never enters the priority score.
- Validation results and derived tables should be saved under `review_outputs/`.
- Reusable implementation should be promoted into the existing `fireline/` package only after the notebook logic is validated.

In [1]:
from pathlib import Path
import csv
import hashlib
import json

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ADDITIONS = ROOT / 'data' / 'additions'
OUTPUT = ROOT / 'review_outputs'
OUTPUT.mkdir(exist_ok=True)
ADDITIONS

WindowsPath('C:/Users/Inaka/Downloads/Fireline-checkpoint3/Fireline-checkpoint3/data/additions')

## Phase 1 — Boundary inventory

In [2]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

boundary_rows = []
boundary_frames = []
for path in sorted(ADDITIONS.glob('*.geojson')):
    frame = gpd.read_file(path).to_crs(4326)
    boundary_frames.append(frame.assign(_source_file=path.name))
    boundary_rows.append({
        'file': path.name,
        'features': len(frame),
        'crs': str(frame.crs),
        'geometry_types': sorted(frame.geometry.geom_type.dropna().unique()),
        'invalid_geometries': int((~frame.geometry.is_valid).sum()),
        'empty_geometries': int(frame.geometry.is_empty.sum()),
        'columns': [c for c in frame.columns if c != 'geometry'],
        'sha256': sha256(path),
    })
boundary_inventory = pd.DataFrame(boundary_rows)
boundary_inventory

,file,features,crs,geometry_types,invalid_geometries,empty_geometries,columns,sha256
0,Provinsi Kalimantan Barat-KAB_KOTA.geojson,14,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",281451c2b73fbf5025c56316ad02302a9eb282f3bd33ce...
1,Provinsi Kalimantan Selatan-KAB_KOTA.geojson,13,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",5eaf5abd0da69a9d27d2e40ee4ce8454f41626474217fa...
2,Provinsi Kalimantan Tengah-KAB_KOTA.geojson,14,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",c42bd23e19bbb8758749b8fe44b66a1545baa8135b066d...
3,Provinsi Kalimantan Timur-KAB_KOTA.geojson,10,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",df89c26d7ce054ec1f390d2a52fa050625c853c3ccf45b...
4,Provinsi Kalimantan Utara-KAB_KOTA.geojson,5,EPSG:4326,[MultiPolygon],0,0,"[kode_kk, kode_prov, kab_kota, provinsi, fid, ...",fdc59f685c71bbf7d384030cb895faba423bdac3a3c9dc...


In [3]:
kabupaten = gpd.GeoDataFrame(pd.concat(boundary_frames, ignore_index=True), crs=4326)
required = {'kode_kk', 'kode_prov', 'kab_kota', 'provinsi', 'geometry'}
missing = required - set(kabupaten.columns)
assert not missing, f'Missing boundary fields: {sorted(missing)}'
assert kabupaten.kode_kk.notna().all()
assert kabupaten.kode_kk.astype(str).is_unique
assert kabupaten.geometry.is_valid.all()
kabupaten = kabupaten.rename(columns={
    'kode_kk': 'kabupaten_code',
    'kode_prov': 'province_code',
    'kab_kota': 'kabupaten_name',
    'provinsi': 'province_name',
})
kabupaten[['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', '_source_file']].head()

,province_code,province_name,kabupaten_code,kabupaten_name,_source_file
0,61,Kalimantan Barat,61.07,Bengkayang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
1,61,Kalimantan Barat,61.02,Mempawah,Provinsi Kalimantan Barat-KAB_KOTA.geojson
2,61,Kalimantan Barat,61.06,Kapuas Hulu,Provinsi Kalimantan Barat-KAB_KOTA.geojson
3,61,Kalimantan Barat,61.71,Kota Pontianak,Provinsi Kalimantan Barat-KAB_KOTA.geojson
4,61,Kalimantan Barat,61.12,Kubu Raya,Provinsi Kalimantan Barat-KAB_KOTA.geojson


## Phase 1 — Population inventory

Population sources have different layouts and units. The next cells should normalize only confirmed kabupaten-level sources. Province totals and unmatched rows remain audit outputs, not silent joins.

In [4]:
population_files = sorted(ADDITIONS.glob('*.csv')) + sorted(ADDITIONS.glob('*.xlsx'))
[(p.name, p.suffix, p.stat().st_size, sha256(p)) for p in population_files]

[('Jumlah Penduduk menurut Kabupaten_Kota di Provinsi Kalimantan Timur, 2024.csv',
  '.csv',
  309,
  '9dbaf2db9ad5fb42634bcec5d208b5b3fe09d6ea37217a4609c031527e7208a1'),
 ('Jumlah Penduduk Menurut Kabupaten_Kota, 2024.csv',
  '.csv',
  328,
  '9df2b10def456a6febe763c60b7cc9df00689d34b280b6d49ceae3e589db91f2'),
 ('Penduduk, Laju Pertumbuhan Penduduk, Distribusi Persentase Penduduk Kepadatan Penduduk, Rasio Jenis Kelamin Penduduk Menurut Kabupaten_Kota di Provinsi Kalimantan Selatan, 2024.csv',
  '.csv',
  1348,
  '9b760daf1fbe4b5cde9f6bcc44a1a562c35b6d380a7f1bd97ecbb8cce5767db2'),
 ('Penduduk, Laju Pertumbuhan Penduduk, Distribusi Persentase Penduduk Kepadatan Penduduk, Rasio Jenis Kelamin Penduduk Menurut Kabupaten_Kota di Provinsi Kalimantan Utara, 2024.csv',
  '.csv',
  1426,
  '76859b29ed4ac2dda69045cd9a19c5d60633ba039b1e5f6d44114b5354a8eb14'),
 ('jumlah-penduduk-menurut-kabupaten-kota-di-provinsi-kalimantan-tengah-ribu-2020-2025(1).xlsx',
  '.xlsx',
  10470,
  '1d3b7f3f7e6f54c552e

## Later phases

1. Normalize administrative IDs and names.
2. Validate point-to-kabupaten attribution, including border cases.
3. Load existing grid-day outputs without changing score calculations.
4. Produce `kabupaten_day_summary`.
5. Prepare the dashboard payload and hierarchical UI behavior.
6. Run regression checks against the current province-level outputs.

## Phase 2 — Canonical administrative model

In [5]:
canonical_fields = ['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', 'geometry', '_source_file']
admin = kabupaten[canonical_fields].copy()
for column in ['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', '_source_file']:
    admin[column] = admin[column].astype(str).str.strip()
admin['admin_id'] = admin['kabupaten_code']
admin['province_code_from_kabupaten'] = admin['kabupaten_code'].str.split('.').str[0]
admin['province_consistent'] = admin['province_code'] == admin['province_code_from_kabupaten']
admin[['admin_id', 'province_code', 'province_name', 'kabupaten_name', 'province_consistent']].head()

,admin_id,province_code,province_name,kabupaten_name,province_consistent
0,61.07,61,Kalimantan Barat,Bengkayang,True
1,61.02,61,Kalimantan Barat,Mempawah,True
2,61.06,61,Kalimantan Barat,Kapuas Hulu,True
3,61.71,61,Kalimantan Barat,Kota Pontianak,True
4,61.12,61,Kalimantan Barat,Kubu Raya,True


In [6]:
expected_provinces = {'61', '62', '63', '64', '65'}
assert set(admin.province_code) == expected_provinces
assert admin.admin_id.is_unique
assert admin[['province_name', 'kabupaten_name']].notna().all().all()
assert admin.province_consistent.all()
assert admin.geometry.notna().all() and admin.geometry.is_valid.all()
assert len(admin) == 56
phase2_validation = pd.DataFrame([{'total_kabupaten_features': len(admin), 'unique_admin_ids': admin.admin_id.nunique(), 'province_count': admin.province_code.nunique(), 'invalid_geometries': int((~admin.geometry.is_valid).sum()), 'province_code_mismatches': int((~admin.province_consistent).sum()), 'duplicate_admin_ids': int(admin.admin_id.duplicated().sum())}])
phase2_validation

,total_kabupaten_features,unique_admin_ids,province_count,invalid_geometries,province_code_mismatches,duplicate_admin_ids
0,56,56,5,0,0,0


In [7]:
admin_lookup = (admin.drop(columns='geometry')[['admin_id', 'province_code', 'province_name', 'kabupaten_name', '_source_file']].rename(columns={'admin_id': 'kabupaten_code'}).sort_values('kabupaten_code').reset_index(drop=True))
admin_lookup.to_csv(OUTPUT / 'kabupaten_admin_lookup.csv', index=False)
admin_lookup

,kabupaten_code,province_code,province_name,kabupaten_name,_source_file
0,61.01,61,Kalimantan Barat,Sambas,Provinsi Kalimantan Barat-KAB_KOTA.geojson
1,61.02,61,Kalimantan Barat,Mempawah,Provinsi Kalimantan Barat-KAB_KOTA.geojson
2,61.03,61,Kalimantan Barat,Sanggau,Provinsi Kalimantan Barat-KAB_KOTA.geojson
3,61.04,61,Kalimantan Barat,Ketapang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
4,61.05,61,Kalimantan Barat,Sintang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
5,61.06,61,Kalimantan Barat,Kapuas Hulu,Provinsi Kalimantan Barat-KAB_KOTA.geojson
6,61.07,61,Kalimantan Barat,Bengkayang,Provinsi Kalimantan Barat-KAB_KOTA.geojson
7,61.08,61,Kalimantan Barat,Landak,Provinsi Kalimantan Barat-KAB_KOTA.geojson
8,61.09,61,Kalimantan Barat,Sekadau,Provinsi Kalimantan Barat-KAB_KOTA.geojson
9,61.10,61,Kalimantan Barat,Melawi,Provinsi Kalimantan Barat-KAB_KOTA.geojson


### Phase 2 result

The canonical model uses `kode_kk` as `kabupaten_code`, `kode_prov` as `province_code`, `kab_kota` as `kabupaten_name`, and `provinsi` as `province_name`. All 56 records pass uniqueness, geometry, and province-prefix checks. Population joins remain a later enrichment step.

## Phase 3 — Grid-day to kabupaten attribution

In [8]:
grid_path = OUTPUT / 'grid_day_priority.csv'
assert grid_path.exists(), f'Missing existing grid output: {grid_path}'
grid_days = pd.read_csv(grid_path)
required_grid_fields = {'grid_id', 'date', 'latitude', 'longitude', 'province_code', 'province_name', 'priority_score'}
missing_grid_fields = required_grid_fields - set(grid_days.columns)
assert not missing_grid_fields, f'Missing grid fields: {sorted(missing_grid_fields)}'
grid_points = gpd.GeoDataFrame(grid_days.copy(), geometry=gpd.points_from_xy(grid_days.longitude, grid_days.latitude), crs=4326)
grid_points.head()

,grid_id,date,detection_count,frp_median_mw,frp_max_mw,frp_missing,high_count,nominal_count,low_count,unknown_count,...,quality_note,activity_points,persistence_points,intensity_points,priority_score,priority_level,daily_rank,region_id,region_name,geometry
0,6:1101,2024-08-01,7,23.080,148.55,0,0,7,0,0,...,Jendela awal kurang dari 7 hari,39.809524,5.0,22.261905,67.0714,P2,1,0,Kalimantan Barat,POINT (110.13029 0.6437)
1,3:1104,2024-08-01,5,14.400,33.44,0,0,5,0,0,...,Jendela awal kurang dari 7 hari,38.285714,5.0,19.880952,63.1667,P2,2,0,Kalimantan Barat,POINT (110.43472 0.30698)
2,7:1097,2024-08-01,6,10.665,25.24,0,0,6,0,0,...,Jendela awal kurang dari 7 hari,39.238095,5.0,17.976190,62.2143,P2,3,0,Kalimantan Barat,POINT (109.75989 0.7637)
3,-3:1119,2024-08-01,4,14.065,25.34,0,0,4,0,0,...,Jendela awal kurang dari 7 hari,36.761905,5.0,19.642857,61.4048,P2,4,0,Kalimantan Barat,POINT (111.92082 -0.24846)
4,4:1105,2024-08-01,3,21.610,44.11,0,0,3,0,0,...,Jendela awal kurang dari 7 hari,33.904762,5.0,22.023810,60.9286,P2,5,0,Kalimantan Barat,POINT (110.57555 0.47567)


In [9]:
# Retain all candidate matches for the audit, then select one deterministic match.
candidates = gpd.sjoin(grid_points, admin[['admin_id', 'province_code', 'province_name', 'kabupaten_name', 'geometry']], how='left', predicate='intersects', lsuffix='grid', rsuffix='admin')
candidate_counts = candidates.groupby(candidates.index)['admin_id'].count()
attribution_ambiguous = candidate_counts.gt(1)
attribution_unmatched = candidate_counts.eq(0)
assigned = (candidates.sort_values(['admin_id'], na_position='last').loc[lambda frame: ~frame.index.duplicated(keep='first')].sort_index())
assigned['kabupaten_assignment_status'] = 'assigned'
assigned.loc[assigned.index.map(attribution_ambiguous).fillna(False), 'kabupaten_assignment_status'] = 'boundary_ambiguous'
assigned.loc[assigned.index.map(attribution_unmatched).fillna(False), 'kabupaten_assignment_status'] = 'unmatched'
assigned['kabupaten_code'] = assigned['admin_id'].fillna('')
assigned['kabupaten_name'] = assigned['kabupaten_name'].fillna('Kabupaten/kota belum terpetakan')
assigned[['grid_id', 'date', 'province_name_grid', 'kabupaten_code', 'kabupaten_name', 'kabupaten_assignment_status']].head()

,grid_id,date,province_name_grid,kabupaten_code,kabupaten_name,kabupaten_assignment_status
0,6:1101,2024-08-01,Kalimantan Barat,61.08,Landak,assigned
1,3:1104,2024-08-01,Kalimantan Barat,61.03,Sanggau,assigned
2,7:1097,2024-08-01,Kalimantan Barat,61.08,Landak,assigned
3,-3:1119,2024-08-01,Kalimantan Barat,61.10,Melawi,assigned
4,4:1105,2024-08-01,Kalimantan Barat,61.03,Sanggau,assigned


In [10]:
# The join must not alter the existing score or row grain.
assert len(assigned) == len(grid_days)
assert assigned.priority_score.reset_index(drop=True).equals(grid_days.priority_score.reset_index(drop=True))
assert assigned[['grid_id', 'date']].duplicated().sum() == 0
assert set(assigned.kabupaten_assignment_status) <= {'assigned', 'boundary_ambiguous', 'unmatched'}
phase3_audit = (assigned.groupby('kabupaten_assignment_status', dropna=False).size().rename('grid_day_rows').reset_index())
phase3_audit

,kabupaten_assignment_status,grid_day_rows
0,assigned,26798


In [11]:
assigned_export = assigned.rename(columns={'province_code_grid': 'province_code', 'province_name_grid': 'province_name'})
attribution_columns = [c for c in grid_days.columns if c in assigned_export.columns]
attributed_grid_days = assigned_export[attribution_columns + ['kabupaten_code', 'kabupaten_name', 'kabupaten_assignment_status']].copy()
attributed_grid_days.to_csv(OUTPUT / 'grid_day_kabupaten_attribution.csv', index=False)
phase3_summary = pd.DataFrame([{
    'grid_day_rows': len(attributed_grid_days),
    'assigned_rows': int((assigned.kabupaten_assignment_status == 'assigned').sum()),
    'boundary_ambiguous_rows': int((assigned.kabupaten_assignment_status == 'boundary_ambiguous').sum()),
    'unmatched_rows': int((assigned.kabupaten_assignment_status == 'unmatched').sum()),
    'unique_kabupaten_assigned': int(assigned.loc[assigned.kabupaten_code.ne(''), 'kabupaten_code'].nunique()),
}])
phase3_summary

,grid_day_rows,assigned_rows,boundary_ambiguous_rows,unmatched_rows,unique_kabupaten_assigned
0,26798,26798,0,0,56


### Phase 3 result

The attribution output preserves the existing grid-day row grain and priority scores. Each row carries a canonical kabupaten code/name plus an explicit assignment status. Ambiguous or unmatched points remain auditable rather than being silently duplicated or dropped.